# Data Processing

In [25]:
# 1. Import Libraries
from pathlib import Path
import pandas as pd

# 2. Dataset Path
DATA_DIR = Path("../../data/raw/COLDataset")

# 3. Load Train and Validation Datasets
train_df = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
validation_df = pd.read_csv(DATA_DIR / "validation.csv", encoding="utf-8-sig")

# 4. Data Preprocessing
for df in [train_df, validation_df]:
    df["TEXT"] = df["TEXT"].fillna("").astype(str).str.strip()
    df["label"] = df["label"].astype(int)


# 5. Split Features and Labels
X_train = train_df["TEXT"]
y_train = train_df["label"]

X_validation = validation_df["TEXT"]
y_validation = validation_df["label"]


# 6. Check Dataset
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

print("\nTrain Label Distribution:")
print(y_train.value_counts())

print("\nValidation Label Distribution:")
print(y_validation.value_counts())

display(train_df[["TEXT", "label"]].head())


Train: (25726, 5)
Validation: (6431, 5)

Train Label Distribution:
label
0    13003
1    12723
Name: count, dtype: int64

Validation Label Distribution:
label
0    3220
1    3211
Name: count, dtype: int64


,TEXT,label
0,也是，想想物流爆仓等半个月和打砸抢烧丧尸围城一般冒着被黑人胖大妈一屁股坐死的风险半夜2点去门...,1
1,说这些。四川人是不是只有喝西北风才健康啊。抛开剂量谈毒性都是垃圾,1
2,不同构成，有绿绿也有天主教，两样都很多的，有真正非洲黑也有南亚那种黑，不过也没有很多，至少不...,0
3,台湾人这么说,0
4,曾泽生肯定是不稳的，想都想得到。人滇军是被强行绑票到东北了。被扔在长春这个铁桶里饿了大半年，...,0


# Character-level TF-IDF + Logistic Regression

In [26]:
# ==========================================
# Character-level TF-IDF + Logistic Regression
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Create TF-IDF Vectorizer
char_vectorizer = TfidfVectorizer(
    analyzer="char",        # 按字符而非词语提取特征
    ngram_range=(2, 5),     # 提取连续 2～5 个字符的组合
    min_df=2,               # 只保留至少出现在两条文档中的特征
    sublinear_tf=True       # 使用对数词频，降低重复词频的影响
)


# 2. Extract Features
X_train_char = char_vectorizer.fit_transform(X_train)
X_validation_char = char_vectorizer.transform(X_validation)

print("Train Feature Shape:", X_train_char.shape)
print("Validation Feature Shape:", X_validation_char.shape)

# 3. Train Logistic Regression
char_model = LogisticRegression(
    max_iter=2000,      # 优化算法允许的最大迭代次数
    random_state=42
)
char_model.fit(X_train_char, y_train)

# 4. Validation Prediction
char_validation_pred = char_model.predict(X_validation_char)

# 5. Evaluation
print("\nCharacter-level TF-IDF Result")
print("Accuracy:", accuracy_score(y_validation, char_validation_pred))

print("\nClassification Report:")
print(classification_report(
    y_validation,
    char_validation_pred,
    target_names=["Safe", "Offensive"],
    digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, char_validation_pred))


Train Feature Shape: (25726, 338474)
Validation Feature Shape: (6431, 338474)

Character-level TF-IDF Result
Accuracy: 0.8815114290157052

Classification Report:
              precision    recall  f1-score   support

        Safe     0.8779    0.8866    0.8823      3220
   Offensive     0.8852    0.8764    0.8808      3211

    accuracy                         0.8815      6431
   macro avg     0.8816    0.8815    0.8815      6431
weighted avg     0.8815    0.8815    0.8815      6431


Confusion Matrix:
[[2855  365]
 [ 397 2814]]


# Word-level TF-IDF + Logistic Regression

In [27]:
# ==========================================
# Word-level TF-IDF + Logistic Regression
# ==========================================
import jieba
import re

# 1. Chinese Tokenizer
def chinese_tokenizer(text):
    words = jieba.lcut(text)       # 中文需要分词器
    return [
        word.strip()
        for word in words
        if re.search(r"[\u4e00-\u9fffA-Za-z0-9]", word)
    ]


# 2. Word-level TF-IDF
word_vectorizer = TfidfVectorizer(
    analyzer="word",           # 按词语处理
    tokenizer=chinese_tokenizer,  # 使用 jieba 分词
    token_pattern=None,     # 禁用默认正则分词规则
    ngram_range=(1, 2),     # 表示字符数量，而是词语数量  unigram 和 bigram
    min_df=2,               # 至少出现在两个文档中
    sublinear_tf=True
)

# 3. Extract Features
X_train_word = word_vectorizer.fit_transform(X_train)
X_validation_word = word_vectorizer.transform(X_validation)

print("Train Feature Shape:", X_train_word.shape)
print("Validatio Feature Shape:", X_validation_word.shape)

# 4. Train Logistic Regression
word_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

word_model.fit(X_train_word, y_train)

# 5. Validation Prediction
word_validation_pred = word_model.predict(X_validation_word)

# 6. Evaluation
print("\nWord-level TF-IDF Result")
print("Accuracy:", accuracy_score(y_validation, word_validation_pred))

print("\nClassification Report:")
print(classification_report(
    y_validation,
    word_validation_pred,
    target_names=["Safe", "Offensive"],
    digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, word_validation_pred))


Train Feature Shape: (25726, 79288)
Validatio Feature Shape: (6431, 79288)

Word-level TF-IDF Result
Accuracy: 0.8897527600684186

Classification Report:
              precision    recall  f1-score   support

        Safe     0.8869    0.8938    0.8903      3220
   Offensive     0.8927    0.8857    0.8892      3211

    accuracy                         0.8898      6431
   macro avg     0.8898    0.8897    0.8897      6431
weighted avg     0.8898    0.8898    0.8898      6431


Confusion Matrix:
[[2878  342]
 [ 367 2844]]


In [28]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (precision_recall_fscore_support, ConfusionMatrixDisplay)

def plot_model_results(y_true, y_pred, model_name):
    labels = ["Safe", "Offensive"]
    x = np.arange(2)

    # 1. Precision / Recall / F1
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0) 
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    width = 0.25

    axes[0].bar(x - width, precision, width, label="Precision")
    axes[0].bar(x, recall, width, label="Recall")
    axes[0].bar(x + width, f1, width, label="F1")
    axes[0].set_xticks(x, labels)
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Classification Metrics")
    axes[0].legend()


    # 2. Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        labels=[0, 1],
        display_labels=labels,
        ax=axes[1],
        colorbar=False
    )
    axes[1].set_title("Confusion Matrix")


    # 3. Actual vs Predicted
    actual = np.bincount(np.asarray(y_true, dtype=int), minlength=2)
    predicted = np.bincount(np.asarray(y_pred, dtype=int), minlength=2)

    axes[2].bar(x - 0.2, actual, 0.4, label="Actual")
    axes[2].bar(x + 0.2, predicted, 0.4, label="Predicted")
    axes[2].set_xticks(x, labels)
    axes[2].set_title("Class Distribution")
    axes[2].legend()

    fig.suptitle(model_name)
    plt.tight_layout()
    plt.show()
